# FAISS Index Construction
**Purpose:** Build a FAISS vector index from the embeddings generated 
in notebook 02. The index enables fast semantic similarity search 
over 1,740 Paris cultural events.

**Input:**
- `data/processed/chunks.json` — event metadata
- `data/processed/embeddings.npy` — 1,740 × 1,024 float32 vectors

**Output:**
- `data/processed/faiss_index.idx` — FAISS binary index
- `data/processed/metadata.json` — metadata parallel to index

**Author:** Hope  
**Date:** 2026-03-23

## 1.1 Imports and environment validation

In [11]:
import json
import numpy as np
import faiss
from pathlib import Path
from dotenv import load_dotenv
from mistralai import Mistral
import os

load_dotenv(dotenv_path=Path("../.env"))

api_key = os.getenv("MISTRAL_API_KEY")
assert api_key is not None, "MISTRAL_API_KEY not found in .env"

print("Environment loaded successfully.")
print(f"FAISS version: {faiss.__version__}")

Environment loaded successfully.
FAISS version: 1.13.2


## 1.2 Configuration

In [4]:
# Cell 2 — 
PROCESSED_DIR = Path("../data/processed")
CHUNKS_PATH   = PROCESSED_DIR / "chunks.json"
EMBED_PATH    = PROCESSED_DIR / "embeddings.npy"
INDEX_PATH    = PROCESSED_DIR / "faiss_index.idx"
META_PATH     = PROCESSED_DIR / "metadata.json"

EMBEDDING_DIM = 1024

print("Configuration set:")
print(f"  Chunks    : {CHUNKS_PATH}")
print(f"  Embeddings: {EMBED_PATH}")
print(f"  Index out : {INDEX_PATH}")
print(f"  Metadata  : {META_PATH}")
print(f"  Dimension : {EMBEDDING_DIM}")

Configuration set:
  Chunks    : ../data/processed/chunks.json
  Embeddings: ../data/processed/embeddings.npy
  Index out : ../data/processed/faiss_index.idx
  Metadata  : ../data/processed/metadata.json
  Dimension : 1024


## 1.3 Load chunks and embeddings

In [7]:
with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
    chunks = json.load(f)

embeddings = np.load(EMBED_PATH).astype("float32")

print(f"Chunks loaded     : {len(chunks)}")
print(f"Embeddings shape  : {embeddings.shape}")
print(f"Embeddings dtype  : {embeddings.dtype}")

assert len(chunks) == embeddings.shape[0], \
    f"Mismatch: {len(chunks)} chunks vs {embeddings.shape[0]} embeddings"
assert embeddings.shape[1] == EMBEDDING_DIM, \
    f"Wrong dimension: {embeddings.shape[1]} expected {EMBEDDING_DIM}"

print("\nAll assertions passed.")

Chunks loaded     : 1740
Embeddings shape  : (1740, 1024)
Embeddings dtype  : float32

All assertions passed.


## 1.4 Build FAISS index

In [12]:
# Using IndexFlatL2 — exact search, no configuration needed
# Appropriate for POC scale (1,740 vectors)

index = faiss.IndexFlatL2(EMBEDDING_DIM)
index.add(embeddings)

print(f"Index type        : {type(index)}")
print(f"Vectors indexed   : {index.ntotal}")
print(f"Embedding dim     : {index.d}")
assert index.ntotal == len(chunks), \
    f"Index count mismatch: {index.ntotal} vs {len(chunks)}"
print("\nIndex built successfully.")

Index type        : <class 'faiss.swigfaiss.IndexFlatL2'>
Vectors indexed   : 1740
Embedding dim     : 1024

Index built successfully.


## Test index with a sample query

In [13]:
# Verify semantic search works before saving
client = Mistral(api_key=api_key)

def embed_query(text: str) -> np.ndarray:
    """Embed a single query text."""
    response = client.embeddings.create(
        model="mistral-embed",
        inputs=[text]
    )
    return np.array([response.data[0].embedding], dtype="float32")

# Test query
query      = "Concert de musique classique à Paris"
query_vec  = embed_query(query)
k          = 5

distances, indices = index.search(query_vec, k)

print(f"Query: '{query}'")
print(f"\nTop {k} results:")
for rank, (idx, dist) in enumerate(zip(indices[0], distances[0]), 1):
    chunk = chunks[idx]
    print(f"\n  {rank}. {chunk['title']}")
    print(f"     City    : {chunk['city']}")
    print(f"     Date    : {chunk['date_range']}")
    print(f"     Distance: {dist:.4f}")

Query: 'Concert de musique classique à Paris'

Top 5 results:

  1. Concert de Printemps France Asie
     City    : Paris
     Date    : Dimanche 8 mars, 15h30
     Distance: 0.4266

  2. Concert
     City    : Paris
     Date    : Dimanche 29 juin 2025, 16h00
     Distance: 0.4280

  3. Les 4 Saisons de Vivaldi, Petite Musique de Nuit de Mozart​
     City    : Paris
     Date    : Mercredi 11 mars, 20h00
     Distance: 0.4296

  4. Les 4 Saisons de Vivaldi, Petite Musique de Nuit de Mozart
     City    : Paris
     Date    : Mardi 7 avril, 20h00
     Distance: 0.4304

  5. Les 4 Saisons de Vivaldi et la Petite Musique de Nuit de Mozart
     City    : Paris
     Date    : Samedi 5 juillet 2025, 20h45
     Distance: 0.4322


## 1.6 Save FAISS index and metadata

In [14]:
# Save FAISS index
faiss.write_index(index, str(INDEX_PATH))
print(f"FAISS index saved : {INDEX_PATH}")

# Save metadata — parallel array matching index positions
metadata = [
    {
        "uid"       : c["uid"],
        "title"     : c["title"],
        "city"      : c["city"],
        "address"   : c["address"],
        "venue"     : c["venue"],
        "date_begin": c["date_begin"],
        "date_end"  : c["date_end"],
        "date_range": c["date_range"],
        "url"       : c["url"],
        "text"      : c["text"]
    }
    for c in chunks
]

with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print(f"Metadata saved    : {META_PATH}")

# Verify
index_reloaded = faiss.read_index(str(INDEX_PATH))
print(f"\nIndex reload test : {index_reloaded.ntotal} vectors — OK")

FAISS index saved : ../data/processed/faiss_index.idx
Metadata saved    : ../data/processed/metadata.json

Index reload test : 1740 vectors — OK
